# FinBERT Evaluation on Gold Commodity Dataset

This notebook evaluates `ProsusAI/finbert` on the Gold Commodity sentiment dataset and logs metrics to the same W&B destination/group used by the other model notebooks.

## 0. Installation


In [ ]:
%%capture
!pip install -U transformers datasets scikit-learn pandas python-dotenv torch

## Device Check


In [ ]:
import subprocess
import torch

print("=== nvidia-smi (if available) ===")
try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout.strip():
        print(res.stdout)
    else:
        print(res.stderr.strip() or "nvidia-smi returned no output")
except FileNotFoundError:
    print("nvidia-smi not found on this machine.")

if torch.cuda.is_available():
    device_type = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_type = "mps"
else:
    device_type = "cpu"

print(f"Detected runtime device: {device_type}")
if device_type == "cuda":
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

## 1. Environment and Reproducibility Setup


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def load_dotenv_walk(start: Path) -> bool:
    for parent in [start, *start.parents]:
        env_file = parent / '.env'
        if env_file.exists():
            load_dotenv(env_file, override=False)
            print(f'Loaded .env from: {parent}')
            return True
    return False

_ = load_dotenv_walk(Path(os.getcwd()))

## 2. W&B Configuration and Gold Dataset Loading


In [ ]:
from datasets import load_dataset

gold_ds = load_dataset('SaguaroCapital/sentiment-analysis-in-commodity-market-gold')
split_name = 'test' if 'test' in gold_ds else 'train'
gold_raw_df = gold_ds[split_name].to_pandas()

sentence_col = next((c for c in ['sentence', 'text', 'content', 'headline'] if c in gold_raw_df.columns), None)
label_col = next((c for c in ['label_text', 'label', 'sentiment', 'target'] if c in gold_raw_df.columns), None)
if sentence_col is None or label_col is None:
    raise ValueError(f'Could not detect sentence/label columns in dataset columns={list(gold_raw_df.columns)}')

labels = gold_raw_df[label_col]
feature = gold_ds[split_name].features.get(label_col)
if hasattr(feature, 'names'):
    label_map = {i: name.lower() for i, name in enumerate(feature.names)}
    label_text = labels.map(lambda x: label_map.get(int(x), str(x).lower()))
else:
    label_text = labels.astype(str).str.strip().str.lower()
    heuristic = {'0': 'negative', '1': 'neutral', '2': 'positive'}
    label_text = label_text.map(lambda x: heuristic.get(x, x))

eval_df = pd.DataFrame({
    'sentence': gold_raw_df[sentence_col].astype(str),
    'label_text': label_text,
})
print(f'Gold eval samples: {len(eval_df)}')
print(eval_df['label_text'].value_counts())

## 3. Load FinBERT and Run Batched Inference


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

MODEL_NAME = 'ProsusAI/finbert'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device_idx = 0 if torch.cuda.is_available() else -1
clf = pipeline(
    task='text-classification',
    model=model,
    tokenizer=tokenizer,
    device=device_idx,
    truncation=True,
)

id2label = model.config.id2label
label_norm = {k: v.lower() for k, v in id2label.items()}
print('Model labels:', label_norm)

def run_finbert_batched(sentences, batch_size=128):
    preds = []
    for i in range(0, len(sentences), batch_size):
        chunk = sentences[i : i + batch_size]
        outputs = clf(chunk, batch_size=batch_size)
        for out in outputs:
            raw = out['label']
            if raw.startswith('LABEL_'):
                idx = int(raw.split('_')[-1])
                raw = id2label.get(idx, raw)
            raw = raw.lower().strip()
            if raw not in {'negative', 'neutral', 'positive'}:
                raw = 'neutral'
            preds.append(raw)
    return preds

pred_labels = run_finbert_batched(eval_df['sentence'].tolist(), batch_size=128)
true_labels = eval_df['label_text'].tolist()

## 4. Evaluation Metrics


In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score, matthews_corrcoef

LABELS = ['negative', 'neutral', 'positive']

metrics = {
    'macro_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='macro', zero_division=0),
    'micro_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='micro', zero_division=0),
    'weighted_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='weighted', zero_division=0),
    'f1_negative': f1_score(true_labels, pred_labels, labels=['negative'], average='macro', zero_division=0),
    'f1_neutral': f1_score(true_labels, pred_labels, labels=['neutral'], average='macro', zero_division=0),
    'f1_positive': f1_score(true_labels, pred_labels, labels=['positive'], average='macro', zero_division=0),
    'accuracy': accuracy_score(true_labels, pred_labels),
    'mcc': matthews_corrcoef(true_labels, pred_labels),
}
report = classification_report(true_labels, pred_labels, labels=LABELS, zero_division=0)

print('=' * 80)
print('FinBERT (ProsusAI/finbert) on Gold Commodity')
print('=' * 80)
print(report)
for k, v in metrics.items():
    print(f'{k}: {v:.4f}')

## 5. Placeholder Save of Best Model

In [ ]:
print('=' * 80)
print('FinBERT (ProsusAI/finbert) on Gold Commodity')
print('=' * 80)
print(report)
for k, v in metrics.items():
    print(f'{k}: {v:.4f}')
print('Note: W&B logging has been disabled.')

## 6. Cleanup

In [ ]:
from pathlib import Path

EXPORT_DIR = Path("experiments/sentiment_agent/saved_models/random_search_best_placeholder")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if "model" in globals() and "tokenizer" in globals():
    model.save_pretrained(EXPORT_DIR)
    tokenizer.save_pretrained(EXPORT_DIR)
    print(f"Saved model/tokenizer to: {EXPORT_DIR}")
else:
    placeholder = EXPORT_DIR / "README_PLACEHOLDER.txt"
    placeholder.write_text(
        "No in-memory model object found. This directory is reserved for the best model export.",
        encoding="utf-8",
    )
    print(f"Created placeholder file: {placeholder}")

## 6. Cleanup


In [ ]:
import gc

for name in ['clf', 'model', 'tokenizer']:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleanup complete.')